# 05 · Model training

| | |
|---|---|
| **入力** | 04 で作った特徴量行列（`data/features/*.npz`） |
| **出力** | 学習済み FNN の重み + 学習履歴 |

3 つの特徴量セットを比較する. それぞれモダリティ構成に合わせた全結合分類器を使う.

* **non-EEG** — モーション + EMG（`FNN_NonEEG`）
* **Hjorth 融合** — EEG Hjorth + モーション + EMG（`FNN_Fusion`）
* **EEG-only** — EEG Hjorth のみ（`FNN_OnlyEEG`）

学習: Adam, 交差エントロピー, 検証精度による Early Stopping.

In [ ]:
import sys
from pathlib import Path

# notebooks/ から実行したときに motion_intent パッケージを import 可能にする
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from motion_intent import config

In [ ]:
import torch
from torch.utils.data import DataLoader

from motion_intent.datasets import FeatureDataset, ConcatFeatureDataset
from motion_intent.models import FNN_OnlyEEG, FNN_NonEEG, FNN_Fusion
from motion_intent.training import fit

## Load features

In [ ]:
subject = 'subjectA'
feat_dir = config.DATA_DIR / 'features'
D = {p: dict(np.load(feat_dir / f'{subject}_{p}.npz')) for p in ('train', 'val', 'test')}
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## non-EEG: motion + EMG

In [ ]:
def loaders(make_ds, bs=64):
    tr = DataLoader(make_ds('train'), batch_size=bs, shuffle=True)
    va = DataLoader(make_ds('val'), batch_size=bs)
    return tr, va

tr, va = loaders(lambda p: FeatureDataset(D[p]['y'], X_motion=D[p]['motion'], X_emg=D[p]['rms']))
m_noneeg = FNN_NonEEG(d_motion=D['train']['motion'].shape[1], d_emg=D['train']['rms'].shape[1])
h_noneeg = fit(m_noneeg, tr, va, epochs=300, device=device)

## Hjorth fusion: EEG Hjorth + motion + EMG

In [ ]:
tr, va = loaders(lambda p: FeatureDataset(
    D[p]['y'], X_eeg=D[p]['hjorth'], X_motion=D[p]['motion'], X_emg=D[p]['rms']))
m_hjorth = FNN_Fusion(D['train']['hjorth'].shape[1],
                      D['train']['motion'].shape[1], D['train']['rms'].shape[1])
h_hjorth = fit(m_hjorth, tr, va, epochs=300, device=device)

## EEG-only

In [ ]:
tr, va = loaders(lambda p: ConcatFeatureDataset(D[p]['y'], D[p]['hjorth']))
m_eeg = FNN_OnlyEEG(d_eeg=D['train']['hjorth'].shape[1])
h_eeg = fit(m_eeg, tr, va, epochs=300, device=device)

## Learning curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, h in [('non-EEG', h_noneeg), ('Hjorth', h_hjorth), ('EEG-only', h_eeg)]:
    ax[0].plot(h.train_loss, label=name)
    ax[1].plot(h.val_acc, label=name)
ax[0].set(title='training loss', xlabel='epoch')
ax[1].set(title='validation accuracy', xlabel='epoch', ylim=(0, 1))
ax[1].axhline(1 / config.N_CLASSES, color='k', ls=':', label='chance')
ax[1].legend(); plt.tight_layout()

## Save weights

In [ ]:
ckpt_dir = config.DATA_DIR / 'checkpoints' / subject
ckpt_dir.mkdir(parents=True, exist_ok=True)
for name, model in [('noneeg', m_noneeg), ('hjorth', m_hjorth), ('eeg', m_eeg)]:
    torch.save(model.state_dict(), ckpt_dir / f'{name}.pth')